## Data Collection

Importing all the different packages

In [1]:
import os
import pandas as pd 
import sys
# Now import your functions

import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'scripts')))

from utils import get_league_data, get_all_player_stats, save_to_data_raw

## Initializing the league by communicating with the API

In [2]:
league = get_league_data(2024) # Calls the get_league_data function in utils

## Creating and Saving Player Dataframe

### Calling the get_all_player_stats 
Call the function to get all the NFL players in the league into one dataframe that contains their stats across the course of the entire season

In [3]:
df_players = get_all_player_stats(league, num_fa=300) # Collects the player stats by calling the get_all_player_stats function

Pulled 300 free agents


### Saving to a CSV
Calls the save_to_data_raw function to save the dataframe in a file within the data/raw folder

In [4]:
save_to_data_raw(df_players, 'player_stats.csv')

File saved to: /Users/justinmckendry/me204-2025-project-justinmckendry/data/raw/player_stats.csv


## Creating and Saving League Specific Draft dataframe

### Getting the data
Calling the espn_request.get_league_draft function to get a json of all league specific draft data from 2024. Save this as a dataframe called df_draft. 

In [34]:
draft = league.espn_request.get_league_draft()
df_draft = pd.DataFrame(draft['draftDetail']['picks'])

### Cleaning the data
There are multiple columns that are empty as those settings are not turned on in our league. Need to rename some of the columns as well to be able to communicate across tables in the database

In [ ]:
# Dropping irrelevant columns
df_draft = df_draft.drop(columns=['bidAmount', 'keeper', 'nominatingTeamId', 'reservedForKeeper', 'tradeLocked'])

# Renaming columns to database convention
df_draft = df_draft.rename(columns={'playerId': 'player_id', 'teamId': 'team_id'})
df_draft

,autoDraftTypeId,id,lineupSlotId,memberId,overallPickNumber,player_id,roundId,roundPickNumber,team_id
0,0,1,2,{REDACTED-ESPN-MEMBER-ID},1,3117251,1,1,3
1,0,2,4,{REDACTED-ESPN-MEMBER-ID},2,4241389,1,2,14
2,0,3,2,{REDACTED-ESPN-MEMBER-ID},3,3929630,1,3,1
3,0,4,0,{REDACTED-ESPN-MEMBER-ID},4,3918298,1,4,4
4,3,5,2,NaN,5,4427366,1,5,8
...,...,...,...,...,...,...,...,...,...
219,3,220,20,NaN,220,4048244,16,10,8
220,1,221,20,NaN,221,4428209,16,11,4
221,1,222,20,NaN,222,15965,16,12,1
222,0,223,20,{REDACTED-ESPN-MEMBER-ID},223,4362887,16,13,14


### Saving to a csv
Call the save_to_data_raw function again to save the draft data as a csv in data/raw folder

In [7]:
save_to_data_raw(df_draft, 'draft_data.csv')

File saved to: /Users/justinmckendry/me204-2025-project-justinmckendry/data/raw/draft_data.csv


## Getting League Specific Team Data

### Getting the data into a dataframe
Loop through a list of league.teams. This is a list of fantasy football teams created within our league. It contains data like wins and losses as well as points_for and points_against. The draft_projected_rank and final_standing are interesting as well.

In [ ]:
teams = league.teams

teams_data = []

for team in league.teams:
    teams_data.append({
        "team_id": team.team_id,
        "team_name": team.team_name,
        "abbrev": team.team_abbrev,
        "division_id": team.division_id,
        "division_name": team.division_name,
        "wins": team.wins,
        "losses": team.losses,
        "ties": team.ties,
        "points_for": team.points_for,
        "points_against": team.points_against,
        "draft_projected_rank": team.draft_projected_rank,
        "final_standing": team.final_standing
    })

df_teams = pd.DataFrame(teams_data)
df_teams


,team_id,team_name,abbrev,division_id,division_name,wins,losses,ties,points_for,points_against,draft_projected_rank,final_standing
0,1,Stretchy Pups Elite,PUPS,0,McKendry Division,10,4,0,1745.0,1459.0,12,4
1,3,South Bay Starr Power,SBSP,1,McDowell Division,5,9,0,1317.0,1439.0,9,8
2,4,Mission Viejo Goat,PAUL,1,McDowell Division,6,8,0,1512.0,1562.0,13,9
3,6,Husky Express,KRIS,1,McDowell Division,10,4,0,1721.0,1378.0,3,2
4,8,Flying Eagles,EGL,0,McKendry Division,6,6,2,1332.0,1417.0,5,5
5,9,Team Sprawl n Brawl,ROWE,0,McKendry Division,5,8,1,1424.0,1623.0,8,10
6,10,Ambler Thighs,AMBL,0,McKendry Division,5,9,0,1365.0,1559.0,10,12
7,11,B Mac,BB,0,McKendry Division,4,10,0,1331.0,1610.0,1,14
8,12,Jean Machine,TM12,1,McDowell Division,9,5,0,1724.0,1440.0,4,1
9,13,The Ralph Dudes,BRD,1,McDowell Division,6,7,1,1474.0,1496.0,2,7


### Saving to a csv

In [33]:
save_to_data_raw(df_teams, 'teams_data.csv')

File saved to: /Users/justinmckendry/me204-2025-project-justinmckendry/data/raw/teams_data.csv
